# Дообучение (continual fine-tuning) готовой mT5-small модели на новых данныхЭто ПРОДОЛЖЕНИЕ обучения уже готовой модели `ismailoviskandar02/uzbek-text-simplifier`(папка `model` в репозитории), а не обучение с нуля.Новые данные: короткие чанки 10-100 слов (`uz_legal_text_5000_final.csv`, колонки`text` / `simplified_text`).**Версия с исправлениями по итогам первого прогона:**- `fp16=False` -- у mT5 известный баг переполнения/NaN loss в fp16 (вы уже  фиксили это при первом обучении на `domain_legal.csv`).- `low_cpu_mem_usage=False` при загрузке модели -- без этого veса  `encoder.embed_tokens.weight` / `decoder.embed_tokens.weight` не  загружаются из чекпоинта (tied-weights баг с safetensors) и  инициализируются случайно, из-за чего модель выдаёт зацикленные повторы.- `processing_class=tokenizer` вместо `tokenizer=tokenizer` в `Seq2SeqTrainer`  -- переименовано в новых версиях `transformers`.- Ручная перезагрузка лучшего чекпоинта после `trainer.train()` -- у  `load_best_model_at_end=True` тот же tied-weights баг вылезает повторно  при внутренней автоперезагрузке, и портит уже обученную модель.**Runtime → T4 GPU**

In [ ]:
!pip install -q transformers datasets evaluate sentencepiece accelerate rouge_score sacrebleu

## 1. Загрузка новых данных (короткие чанки 10-100 слов)

In [ ]:
from google.colab import files
import os

needed = ['uz_legal_text_5000_final.csv']
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    print('Загрузи файлы:', missing)
    uploaded = files.upload()

# Опционально: загрузи также кусок старого датасета для подмешивания
# (см. ячейку "Подмешивание старых данных" ниже) -- если файла нет, шаг просто пропускается.
OLD_DATA_FILE = 'output.csv'  # поменяй на реальное имя своего исходного датасета, если хочешь подмешать

In [ ]:
import pandas as pd

new_df = pd.read_csv('uz_legal_text_5000_final.csv')[['text', 'simplified_text']]
new_df = new_df.dropna(subset=['text', 'simplified_text'])
new_df = new_df[new_df['text'].str.strip() != '']
new_df = new_df[new_df['simplified_text'].str.strip() != '']
new_df = new_df.drop_duplicates(subset=['text'])

print('Новых примеров (короткие чанки):', len(new_df))
new_df.head(2)

### Подмешивание старых данных (рекомендуется)Если дообучать ТОЛЬКО на коротких чанках, модель рискует "разучиться"работать с длинными текстами, которые составляли основной датасет(catastrophic forgetting). Подмешай случайную выборку старых данных(например 20-30% от объёма новых) -- модель увидит оба распределениядлин на каждой эпохе.Если такого файла нет под рукой -- просто не загружай `OLD_DATA_FILE`,следующая ячейка тогда работает только с новыми данными(`frames = [new_df]`).

In [ ]:
frames = [new_df]

if os.path.exists(OLD_DATA_FILE):
    old_df = pd.read_csv(OLD_DATA_FILE)[['text', 'simplified_text']]
    old_df = old_df.dropna(subset=['text', 'simplified_text'])
    old_df = old_df[old_df['text'].str.strip() != '']
    old_df = old_df[old_df['simplified_text'].str.strip() != '']

    # берём случайную выборку ~30% от объёма новых данных, чтобы новые
    # короткие чанки не потерялись на фоне старого объёма
    sample_size = min(len(old_df), int(len(new_df) * 0.3))
    old_sample = old_df.sample(n=sample_size, random_state=42)
    frames.append(old_sample)
    print(f'Подмешано {len(old_sample)} старых примеров для защиты от catastrophic forgetting')
else:
    print(f'{OLD_DATA_FILE} не найден -- дообучаем только на новых данных.')

df = pd.concat(frames, ignore_index=True)
df = df.drop_duplicates(subset=['text'])

MAX_CHARS = 3000
df = df[(df['text'].str.len() <= MAX_CHARS) & (df['simplified_text'].str.len() <= MAX_CHARS)]

print('Итого примеров для дообучения:', len(df))
df.head(2)

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

train_df, val_df = train_test_split(df, test_size=0.05, random_state=42)

ds = DatasetDict({
    'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.reset_index(drop=True)),
})
print(ds)

## 2. Загрузка ГОТОВОЙ модели (не google/mt5-small!)Модель лежит в подпапке `model` репозитория -- отсюда `subfolder="model"`.`low_cpu_mem_usage=False` -- обязательно, иначе `encoder.embed_tokens.weight`и `decoder.embed_tokens.weight` не подгружаются из чекпоинта (баг сtied-weights при загрузке через meta-device) и заменяются случайнымивесами. Проверяем это ниже через std эмбеддингов.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "ismailoviskandar02/uzbek-text-simplifier"
SUBFOLDER = "model"

# Тексты короткие (10-100 слов, ~150-250 токенов с учётом mT5 SentencePiece) --
# длину контекста можно уменьшить относительно исходного обучения (было 256).
# Если подмешиваешь старые длинные данные, лучше оставить 256.
MAX_INPUT_LEN = 192
MAX_TARGET_LEN = 192
PREFIX = "simplify: "  # тот же префикс, что при первом обучении -- не меняй

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, subfolder=SUBFOLDER)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, subfolder=SUBFOLDER, low_cpu_mem_usage=False
)
model.config.tie_word_embeddings = False  # чекпоинт хранит shared/lm_head отдельно -- убираем лишний warning

# sanity-check: эмбеддинги должны быть ОБУЧЕННЫМИ, не случайными
emb = model.get_input_embeddings().weight
print("embedding mean/std:", emb.mean().item(), emb.std().item())
print("(std должен быть в районе 10+ -- если ~0.02-0.05, эмбеддинги случайные, что-то не так)")

In [ ]:
def preprocess(batch):
    inputs = [PREFIX + t for t in batch['text']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True)
    labels = tokenizer(text_target=batch['simplified_text'], max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized = ds.map(preprocess, batched=True, remove_columns=ds['train'].column_names)

## 3. Метрики (без изменений)

In [ ]:
import evaluate
import numpy as np

rouge = evaluate.load('rouge')
chrf = evaluate.load('chrf')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    chrf_result = chrf.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    result['chrf'] = chrf_result['score']
    return {k: round(v, 4) if isinstance(v, float) else v for k, v in result.items()}

## 4. Дообучение- `learning_rate=1e-4` (ниже, чем `3e-4` в первом обучении) -- продолжаем  обучение уже сошедшейся модели.- `num_train_epochs=3` -- если по логам видно, что chrF всё ещё растёт к  последней эпохе (как было в прошлом прогоне: 72.6 → 73.8 → 75.2, без  признаков плато), смело увеличивай до 5-6 и перезапускай эту ячейку --  `load_best_model_at_end` всё равно вернёт лучший чекпоинт, а не последний.- `fp16=False` -- ОБЯЗАТЕЛЬНО для mT5, известный баг с NaN loss в fp16.

In [ ]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import torch

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir='mt5_uz_simplify_continued',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.02,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model='chrf',
    fp16=False,  # КРИТИЧНО для mT5 -- вызывает NaN loss при включении
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    processing_class=tokenizer,  # было tokenizer= в старых версиях transformers
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
train_result = trainer.train()

# ВАЖНО: load_best_model_at_end=True заставляет Trainer самостоятельно
# перегрузить "лучший" чекпоинт внутри себя -- но делает это СВОИМ способом,
# без low_cpu_mem_usage=False, которым мы чинили баг с tied-эмбеддингами
# при первой загрузке. Тот же баг (missing keys: encoder/decoder.embed_tokens)
# бьёт повторно именно на этом шаге и портит уже обученную модель, если
# его не исправить здесь.
best_ckpt = trainer.state.best_model_checkpoint
print("Лучший чекпоинт:", best_ckpt)

model = AutoModelForSeq2SeqLM.from_pretrained(best_ckpt, low_cpu_mem_usage=False)
model.config.tie_word_embeddings = False
model.to(trainer.args.device)

emb = model.get_input_embeddings().weight
print("embedding mean/std (после перезагрузки):", emb.mean().item(), emb.std().item())
print("(std должен остаться в районе 10+ -- если упал к ~0.02-0.05, баг снова сработал)")

trainer.model = model  # чтобы дальнейшее сохранение шло уже с исправленной моделью

## 5. Сравнение ДО/ПОСЛЕ на коротких текстахПроверяем именно то, что должно было улучшиться -- симплификациюкоротких (10-30 слов) юридических формулировок. Если увидишь зацикленныеповторы слов (`olish, olish, olish...`) -- значит перезагрузка чекпоинтав предыдущей ячейке не сработала, эмбеддинги снова случайные.

In [ ]:
def simplify(text, max_new_tokens=150):
    inputs = tokenizer(PREFIX + text, return_tensors='pt', truncation=True, max_length=MAX_INPUT_LEN).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

short_val = val_df[val_df['text'].str.split().str.len() <= 25].head(3)

for _, row in short_val.iterrows():
    print('ORIGINAL:  ', row['text'])
    print('SIMPLIFIED:', simplify(row['text']))
    print('REFERENCE: ', row['simplified_text'])
    print('---')

## 6. Сохранение и скачивание

In [ ]:
trainer.save_model('mt5_uz_simplify_continued_final')
tokenizer.save_pretrained('mt5_uz_simplify_continued_final')

!zip -r mt5_uz_simplify_continued_final.zip mt5_uz_simplify_continued_final
from google.colab import files as colab_files
colab_files.download('mt5_uz_simplify_continued_final.zip')

## 7. Обновление модели на Hugging Face HubПушим в тот же репозиторий, в ту же подпапку `model` -- иначе Gradio-интерфейс,настроенный на путь `model/`, перестанет находить файлы. Это создаст новуюверсию (коммит) поверх текущей задеплоенной модели -- откатить можно вистории коммитов репозитория.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# trainer.push_to_hub(
#     'ismailoviskandar02/uzbek-text-simplifier',
#     commit_message='Continue fine-tuning on short (10-100 word) legal chunks',
# )
# # если push_to_hub не поддерживает subfolder напрямую -- альтернативно
# # загрузи файлы вручную через huggingface_hub.upload_folder(..., path_in_repo='model')